In [9]:
import requests
import pandas as pd
from google.colab import userdata

# 1. Retrieve the token securely from Colab Secrets
apify_token = userdata.get('APIFY_TOKEN')

# 2. Construct the API URL using the hidden token
api_url = f'https://api.apify.com/v2/datasets/l8rBqhwEaIVpGFJAe/items?token={apify_token}'

# 3. Fetch data
response = requests.get(api_url)
if response.status_code == 200:
    data = response.json()
    df = pd.DataFrame(data)
    print(f'Successfully loaded {len(df)} items.')
    display(df.head())
else:
    print(f'Failed to fetch data: {response.status_code}')
    print(response.text)

Successfully loaded 3 items.


,resultType,batchId,url,status,data,completedAt
0,result,batch_1,https://shopee.co.th/product/1752450934/509594...,completed,"{'bff_meta': None, 'error': None, 'error_msg':...",2026-07-23T10:23:19.976Z
1,result,batch_1,https://shopee.co.th/product/121969451/2033510...,completed,"{'bff_meta': None, 'error': None, 'error_msg':...",2026-07-23T10:23:19.977Z
2,result,batch_1,https://shopee.co.th/product/121969451/1918931...,completed,"{'bff_meta': None, 'error': None, 'error_msg':...",2026-07-23T10:23:19.977Z


In [10]:
# Parse the nested 'data' column
data_parsed = pd.json_normalize(df['data'])

# Based on your provided snippet, the title key is simply 'title' within the item dictionary
# Map the actual nested column names to your requested labels
columns_to_extract = {
    'data.item.item_id': 'item_id',
    'data.item.shop_id': 'shop_id',
    'data.item.title': 'title',
    'data.item.price': 'price',
    'data.item.price_before_discount': 'price_before_discount'
}

# Filter and rename columns safely
available_cols = [c for c in columns_to_extract.keys() if c in data_parsed.columns]
final_df = data_parsed[available_cols].rename(columns=columns_to_extract)

# Convert prices (divide raw integer price by 100,000)
if 'price' in final_df.columns:
    final_df['price'] = final_df['price'] / 100000.0
if 'price_before_discount' in final_df.columns:
    final_df['price_before_discount'] = final_df['price_before_discount'] / 100000.0

# Save to CSV
output_csv_path = 'shopee_products.csv'
final_df.to_csv(output_csv_path, index=False, encoding='utf-8-sig')

print(f"Extracted {len(final_df)} items saved to '{output_csv_path}'")
display(final_df)

Extracted 3 items saved to 'shopee_products.csv'


,item_id,shop_id,title,price,price_before_discount
0,50959436125,1752450934,ไฮยีน น้ำยาซักผ้ามิลค์กี้ทัช Hygiene Wash Milk...,65.0,65.0
1,20335106536,121969451,Fineline ไฟน์ไลน์ ผลิตภัณฑ์ซักผ้าถนอมผ้า พลัสซ...,53.0,89.0
2,19189312571,121969451,ไฟน์ไลน์ซักผ้า กลิ่นซันนี่โกลด์ สูตรลดกลิ่นอับ...,125.0,179.0
